# before running this

- need to update `reslice` in `tissue_extractor` (push to repo)

- restart kernel

- test `reslice_loop` to make sure it reads tissue path if required (for img_path) and t1 path if not

- go through all the code and make sure it is correct.

# MNISymm_coreg_normalize_pipeline

This is an "indirect" pipeline, in that each image is coregistered *before* being normalized to template space.

**Pipeline**:

- coregister T1 anatomical (entire brain + extra-brain tissue) image to reference

- for each subject-week, get a cerebellar isolation mask (this is its own thing - can be used in others. So have this in its own notebook)

- for each subject-week, get normalization files for MNISymm template space

- for each subject-week, normalize into MNI (symmetric):
    
    - white matter segmentation

    - grey matter segmentation

    - T1 anatomical

**NOTE**: this normalization will be done with respect to each week (i.e. reslice using the deformation file from each individual week, NOT the reference week).

### NOTE

this pipeline uses the updated version of SUITPy (from master branch; master branch up-to-date with developer branch).

As of June 15 (at 6:39pm), used updated SUITPy for isolation masks.

1. Coregister each subject's T1 anatomical to their reference week - take their first measurement week as the reference week (SPM: use `sc_anat`).

2. Using the cerebellar isolation mask for each subject-week, get their normalization files in MNISymm space.

3. Normalize (T1_anat, wm, gm segmentations) into symmetric space (for each subject-week)

In [ ]:
# need path to root directory
import sys
sys.path.append('/home/UWO/mporwal2/Documents/GitHub/smarts_cerebellum/')

In [ ]:
# Imports

# model libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# image reading libraries
import nibabel as nib
from nilearn import plotting as npl
import ants

import SUITPy as suit
import SUITPy.atlas as atlas

import nitools as nt
from image_processing import tissue_extractor as te

from pathlib import Path
import os

In [ ]:
# directories
anat_dir = '/cifs/diedrichsen/data/smarts_cerebellum/anatomicals'
p_df = pd.read_csv('/cifs/diedrichsen/data/smarts_cerebellum/participants_anat.tsv', sep = '\t')

## Get normalization files for each subject-week

In [ ]:
# write normalization files for each subject-week (full-image coregistered)
# this just takes the t1_anatomicals and the isolation mask for each subject.


#_______________________________
# base loop
for i in range(0, p_df.shape[0]):
    p_id = p_df['ID'].iloc[i]
    week = (p_df['Week'].iloc[i]).strip() # sometimes have extra white spaces
    p_centre = (str(p_df['Centre'].iloc[i])).strip()

    subj_id = f'{p_centre.strip()}_{p_id}'

    t1_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1.nii'
    mask_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1_cerebellum_dseg.nii.gz'



    # check that paths exist
    if not Path(t1_path).is_file():
        print(f'T1 path does not exist for {subj_id} in week {week}')
        continue

    if not Path(mask_path).is_file():
        print(f'T1 path does not exist for {subj_id} in week {week}')
        continue
    

    # this is a new folder for each subject-week
    results_path = f'{anat_dir}/MNISymm/full_img_coreg/{subj_id}/{week}/'
    results_path.mkdir(parents=True, exist_ok = True)
    #__________________________________

    # function goes here

    te.normalize(t1_path, mask_path, results_path, space = 'MNI152NLin2009cSymC')

    print(f'{subj_id} {week} normalization done')


# store in anat_dir/MNISymm/full_img_coreg/subj_id/week for each subject-week

### Function for reslice loop (this is temporarily in lieu of a proper subject-loop function)

# before we run this

Need to fix tissue extractor `reslice`:

1. Should default `tissue = None`, and have an if-else loop for naming the image with tissue. Or else we can have (as input in this te.reslice function) suffix for name of the output image (in place of "tissue")

### fix for reslice function in tissue extractor

In [ ]:
def reslice(img_path, fwd_def,  mask_path,
            results_path, subj_id,
            suffix, # IF T1, WILL NEED TO SPECIFY HERE TO HAVE IT INCLUDED IN IMAGE NAME
            week = None):
    """
    Inputs:
        img_path (str): path to image being normalized
        fwd_def (str): path to relevant forward deformation file
        mask_path (str): path to cerebellar isolation mask
        results_path (str): path to directory to store normalization files
        subj_id (str)
        suffix (str): suffix for output file name
    """
    if week: # if reslicing images by week
        resliced_img = suit.reslice_image(source_image = img_path,
                                      deformation = fwd_def,
                                      mask = str(mask_path)
                                      )
        nib.save(resliced_img, Path(results_path)/f'{subj_id}_{week}_{suffix}_normalized.nii.gz')

        resliced_img_path = f'{results_path}/{subj_id}_{week}_{suffix}_normalized.nii.gz'

    else: # week == None (just reslice one img per subj) (e.g. slope reslice)
        # for the slope: just put slope file suffix as tissue (i.e. _wm_native_slope_resliced.nii.gz)
        resliced_img = suit.reslice_image(source_image = img_path,
                                      deformation = fwd_def,
                                      mask = str(mask_path)
                                      )
        nib.save(resliced_img, Path(results_path)/f'{subj_id}_{suffix}_normalized.nii.gz')

        resliced_img_path = f'{results_path}/{subj_id}_{suffix}_normalized.nii.gz'

    return resliced_img_path

### done

In [ ]:
def reslice_loop(
        sub_dir, # e.g. MNISymm_T1
        suffix, # name of normalized image
        tissue = None
        ):
    
    tissue_dict = {
        'gm': 'c1',
        'wm': 'c2',
        'csf': 'c3'
    }

    # base loop _____________________________________
    for i in range(0, p_df.shape[0]):
        p_id = p_df['ID'].iloc[i]
        week = (p_df['Week'].iloc[i]).strip() # sometimes have extra white spaces
        p_centre = (str(p_df['Centre'].iloc[i])).strip()
        
        subj_id = f'{p_centre.strip()}_{p_id}'

        # required files for reslice
        if not tissue == None:
            img_path = f'{anat_dir}/{subj_id}/{week}/{tissue_dict[tissue]}{subj_id}_{week}_T1.nii'
        else:
            img_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1.nii'
        print(f'using {img_path}')

        # test_______________________

        

        """
        mask_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1_cerebellum_dseg.nii.gz'
        fwd_def = f'{anat_dir}/{subj_id}/{week}/MNISymm/full_img_coreg/{subj_id}_{week}_T1_to-MNI152NLin2009cSymC_mode-image_xfm.nii.gz'

        # check that paths exist
        if not Path(img_path).is_file():
            print(f'T1 path does not exist for {subj_id} in week {week}')
            continue
            
        if not Path(mask).is_file():
            print(f'mask path does not exist for {subj_id} in week {week}')
            continue
        """
        
        """
        # CHECK THIS PART using one subject
        results_path = f'{sub_dir}/{subj_id}/{week}'
        results_path.mkdir(parents=True, exist_ok = True)
        """

        """
        # we will use the week option in reslice, so it will save each week's resliced image to each week's directory
        te.reslice(img_path = img_path,
                   fwd_def = fwd_def,
                   mask_path = mask_path,

                   results_path = results_path,
                   subj_id = subj_id,
                   suffix = suffix,
                   week = week)
        
        if not tissue == None:
            print(f'Normalization done for {subj_id} {week} for {tissue}')
        else:
            print(f'Normalization done for {subj_id} at {week} for T1 anatomical')
        """

        



When reslicing segmentation (tissue) files: suffix = wm_MNISymm
When reslicing T1 anatomical files: suffix = T1_MNISymm

("normalized" is already included as the actual suffix for the image, so this would be {suffix}_normalized)

So we have three folders: MNISymm_<type> where type = T1, wm, gm

In each of these folders, we will save the resliced t1 anatomical or tissue file for each subject-week.

So it will have: subject --> week --> normalized file (for each of these three folders) - so one file per subject week.

We can probably collapse this into just subjects and have all of their week files in the same directory.

## Normalize the images to template (MNISymm) space

In [ ]:
# normalize T1 anatomicals to MNISymm template space
reslice_loop(sub_dir = "MNISymm_T1",
             suffix = 'T1_MNISymm')

In [ ]:
# normalize wm segmentations to MNISymm template space
reslice_loop(sub_dir = "MNISymm_T1",
             suffix = 'T1_MNISymm')